# 18.13 圖片／音訊能力能如何蒸餾？


教師看圖時用16個視覺token，學生只用4個，回答三個文字token。若把兩條整段logits直接對上，同一列可能一邊仍在處理圖片、另一邊已經在回答，機率就不具相同意義。前置是[模態展開後的位置](https://birdhackor.github.io/tiny-perceptron-vlm/10.7.html)、[回答第一token的預測位置](https://birdhackor.github.io/tiny-perceptron-vlm/7.4.html)與[分布對齊](https://birdhackor.github.io/tiny-perceptron-vlm/18.6.html)。蒸餾可以比較相同答案，但需要先找到各自真正負責預測答案的位置。

用沒有角色或問題額外位置的玩具序列，教師前文16格、學生4格，答案為A0、A1、A2。A0輸入位置分別16與4，但預測A0的logits來自它前一格，也就是15與3；A1由16與4預測，A2由17與5預測。這個向左一格的對齊非常重要，不能因只在說圖片就忘記下一字預測的規則。

圖中每條橫線連接相同「待預測答案」，不是相同絕對索引。第一列教師query15與學生query3都預測A0，即使前文長度不同，也能比較同一詞表分布。下列兩列同理；答案輸入本身則從16/4開始，和圖裡的Query位置分開。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/architecture_modal_answer_alignment.svg")))

In [ ]:
import torch
from tiny_perceptron.alignment import distillation_kl

teacher_prefix, student_prefix, answer_tokens = 16, 4, 3
t_rows = list(range(teacher_prefix - 1, teacher_prefix + answer_tokens - 1))
s_rows = list(range(student_prefix - 1, student_prefix + answer_tokens - 1))
t_logits = torch.zeros(1, teacher_prefix + answer_tokens, 2)
s_logits = torch.zeros(1, student_prefix + answer_tokens, 2)
t_logits[:, t_rows] = torch.tensor([0.8, 0.2]).log()
t_answer, s_answer = t_logits[:, t_rows], s_logits[:, s_rows]
labels = torch.zeros(1, answer_tokens, dtype=torch.long)
print("預測位置", t_rows, s_rows)
print("對齊形狀", tuple(t_answer.shape), tuple(s_answer.shape))
print("答案KL", round(distillation_kl(s_answer, t_answer, labels, temperature=1).item(), 4))

輸入只是人工分數表，沒有載入圖片或聲音。教師選中三格都設比例 `[0.8,0.2]`，學生零分數則均勻；`t_rows`和`s_rows`各自挑出預測相同答案的logits，labels將三格都標成有效。應印出 `[15,16,17]`與 `[3,4,5]`，兩邊形狀同為 `(1,3,2)`，KL0.1927。未對齊時整段長度19與7不同，不能直接按列做分布誤差。

真實模型還含角色、問題、圖像標記或音訊框，答案位置應從展開後的有效labels與預測位移推導，不能固定套16和4。兩邊也需看同一份圖片/聲音與問題，否則學生是在學另一個條件下的教師回答。若比較內部視覺特徵，因數量與維度不同，還需另外設計映射與目標，不是直接複用這個文字KL。

練習只把student_prefix改成6，保持同樣三個答案。先預測s_rows變 `[5,6,7]`，教師不變，對齊後形狀與KL仍相同，再執行核對。音訊前文框數變化也遵循同樣道理：找到語意相同的答案位置，再比較分布。
